In [40]:
import json
from pathlib import Path
from collections import Counter

In [41]:
IN_JSONL = Path("/Users/varunchandrashekar/Tenantmate/code/data/processed/nsw_sections.jsonl")
OUT_JSONL = Path("/Users/varunchandrashekar/Tenantmate/code/data/processed/nsw_chunks.jsonl")

with open(IN_JSONL) as f_in, open(OUT_JSONL, "w") as f_out:
    for line in f_in:
        record = json.loads(line)
        if record.get("schedule"):
            sched_tag = "".join(record["schedule"].split()[:2])   # "Schedule1"
            record["chunk_id"] = f"NSW-RTA2010-{sched_tag}-s{record['section_number']}"
        else:
            record["chunk_id"] = f"NSW-RTA2010-s{record['section_number']}"
        f_out.write(json.dumps(record, ensure_ascii=False) + "\n")

In [42]:
#Verfying the stored output
with open(OUT_JSONL) as f:
    chunks = [json.loads(line) for line in f]

print(f"Total chunks: {len(chunks)}")
print(f"\nFirst chunk_id: {chunks[0]['chunk_id']}")
print(f"Sample suffix:  {chunks[10]['chunk_id']}")


Total chunks: 328

First chunk_id: NSW-RTA2010-s1
Sample suffix:  NSW-RTA2010-s11


In [43]:
#Checking if there are any duplicates in the tags
ids = [c["chunk_id"] for c in chunks]
print(f"Unique IDs: {len(set(ids))} / {len(ids)}")

Unique IDs: 328 / 328


In [44]:
id_counts = Counter(c["chunk_id"] for c in chunks)
collisions = {cid: n for cid, n in id_counts.items() if n > 1}
print(f"Colliding chunk_ids: {len(collisions)}\n")

for cid in collisions:
    print(f"=== {cid} appears {collisions[cid]} times ===")
    matches = [c for c in chunks if c["chunk_id"] == cid]
    for m in matches:
        print(f"  Title:    {m['section_title'][:80]}")
        print(f"  Part:     {m['part']}")
        print(f"  Schedule: {m.get('schedule')}")
        print(f"  First 100: {m['text'][:100]}\n")

Colliding chunk_ids: 0



In [45]:
counts = Counter(c["section_number"] for c in chunks)
dupes = {num: count for num, count in counts.items() if count > 1}
print(f"Dupicate section numbers: {len(dupes)}\n")

Dupicate section numbers: 36



In [46]:
for num in list(dupes)[:5]:
    print(f"Section number '{num}' appeards {dupes[num]} times")
    matches = [c for c in chunks if c["section_number"] == num]
    for m in matches:
        print(f"  Title: {m['section_title'][:80]}")
        print(f"  Part:  {m['part']}")
        print(f"  First 100 chars: {m['text'][:100]}\n")


Section number '1' appeards 3 times
  Title: Name of Act
  Part:  Part 1 Preliminary
  First 100 chars: This Act is the Residential Tenancies Act 2010.

  Title: Definitions
  Part:  Part 1 General
  First 100 chars: In this Schedule—
appointed member means a member appointed by the Minister under section 178(1)(d).

  Title: Regulations
  Part:  Part 1 General
  First 100 chars: (1) The regulations may contain provisions of a savings or transitional nature consequent
on the ena

Section number '2' appeards 3 times
  Title: Commencement
  Part:  Part 1 Preliminary
  First 100 chars: This Act commences on a day or days to be appointed by proclamation.

  Title: Terms of office of members
  Part:  Part 2 Constitution
  First 100 chars: Subject to this Schedule and the regulations, an appointed member holds office for such
period (not 

  Title: Definitions
  Part:  Part 2 Provisions consequent on enactment of this Act
  First 100 chars: In this Part—
existing residential tenancy agreemen